<a href="https://colab.research.google.com/github/jnlyu6/Diehl_Cook_2015_Replication/blob/main/Import_data_folder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title ## ***Import folder from GitHub public repo***

import os
import shutil
import subprocess

from pathlib import Path
from collections import defaultdict

def import_data_from_git(repo_url, temp_dir, target_folder):
    try:
        # 1. Clean up any stale temp directories from previous failed attempts
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

        # 2. Clone the repository to the temporary directory
        print(f"Cloning {repo_url} into temporal storage...")
        subprocess.run(["git", "clone", repo_url, temp_dir], check=True)

        # 3. Check if the target folder exists inside the cloned repository
        repo_target_path = os.path.join(temp_dir, target_folder)
        if os.path.exists(repo_target_path):
            # Remove the existing local data directory if it exists
            if os.path.exists(target_folder):
                print(f"Removing existing '{target_folder}' directory to avoid conflicts...")
                shutil.rmtree(target_folder)

            # Copy the fresh folder to the workspace
            print(f"Copying '{target_folder}' folder to workspace...")
            shutil.copytree(repo_target_path, target_folder)
            print(f"✅ Successfully imported '{target_folder}' folder!")
        else:
            print(f"❌ Error: The directory '{target_folder}' was not found in the cloned repository.")

    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to clone the repository. Technical details: {e}")
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")
    finally:
        # 4. Ensure cleanup of the temporary repository files to free space
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
            print("Temporary files cleaned up.")

def reassemble_split_files(local_data_path):
    # Reassemble split chunk files (.chunk_* or .part*) back into their original form
    if local_data_path.exists():
        # Handle general .chunk_ split files with markers
        markers = list(local_data_path.glob('*.split_marker'))
        if markers:
            print("\nFound split file chunks with markers. Reassembling...")
            for marker in markers:
                original_name = marker.name.replace('.split_marker', '')
                original_file = local_data_path / original_name
                chunks = sorted(list(local_data_path.glob(f"{original_name}.chunk_*")))
                if chunks:
                    print(f" - Reassembling {original_name} from {len(chunks)} chunks...")
                    with open(original_file, 'wb') as outfile:
                        for chunk in chunks:
                            with open(chunk, 'rb') as infile:
                                outfile.write(infile.read())
                            chunk.unlink()
                    marker.unlink()
                    print(f"   Reassembled successfully!")

        # Specifically handle GitHub auto-split .partX files (like checkpoint_w_hist.npy)
        part_files = sorted(list(local_data_path.glob('*.npy.part*')))
        if part_files:
            # Group part files by their base name (e.g. checkpoint_w_hist.npy)
            grouped_parts = defaultdict(list)
            for p_file in part_files:
                base_name = p_file.name.split('.part')[0]
                grouped_parts[base_name].append(p_file)

            print("\nFound .part split files. Reassembling...")
            for base_name, chunks in grouped_parts.items():
                chunks = sorted(chunks, key=lambda x: int(x.name.split('.part')[-1]))
                original_file = local_data_path / base_name
                print(f" - Reassembling {base_name} from {len(chunks)} parts...")
                with open(original_file, 'wb') as outfile:
                    for chunk in chunks:
                        with open(chunk, 'rb') as infile:
                            outfile.write(infile.read())
                        chunk.unlink()
                print(f"   Reassembled {base_name} successfully!")

def list_local_files(local_data_path):
    if local_data_path.exists():
        print("\nCurrent local files in the 'data/' directory:")
        for f in local_data_path.glob('*'):
            print(f" - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")

    for f in local_data_path.glob('*.part*'):
        if f.exists():
            f.unlink()

# Configure the public repository URL and target folder here
REPO_URL = "https://github.com/jnlyu6/Diehl_Cook_2015_Replication" # @param {type:"string"}
TEMP_DIR = "/tmp/repo_temp"
TARGET_FOLDER = "data" # @param {type:"string"}

# Check if local /content/data exists
local_data_path = Path('data')
has_data = local_data_path.exists() and any(local_data_path.iterdir())

if has_data:
    print("[INFO] Local /content/data directory already exists and is not empty. To prevent overwriting your current training progress, restoration has been safely aborted.")
else:
    # Run the import function
    import_data_from_git(REPO_URL, TEMP_DIR, TARGET_FOLDER)
    reassemble_split_files(local_data_path)
    list_local_files(local_data_path)

[INFO] Local /content/data directory already exists and is not empty. To prevent overwriting your current training progress, restoration has been safely aborted.
